# Restricted-input method reference
Source: analysis/inter-park/blue_green_landuse_analysis.ipynb
Requires independently obtained inputs and source directory layout. Not an end-to-end pipeline. Outputs can contain sensitive data; keep the working directory private. External absolute paths have been replaced by external_inputs/<basename>. See docs/REPRODUCIBILITY.md.


In [ ]:
import os
from pathlib import Path
private_root = Path(os.environ["UFUG_PRIVATE_ROOT"]).resolve()
os.chdir(private_root)


In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd

import os
# Working directory is explicitly configured in the first cell.


In [ ]:
landuse_file = 'external_inputs/landuse_fixed.geojson'

landuse_sg = gpd.read_file(landuse_file)
landuse_sg = landuse_sg.to_crs(epsg=3414)
landuse_sg.plot()



In [ ]:
park_boundaries_path = "gis_files/parks_allthree_boundaries.geojson"

park_boundaries_gdf = gpd.read_file(park_boundaries_path)
park_boundaries_gdf = park_boundaries_gdf.to_crs(epsg=3414)
park_boundaries_gdf.head()
park_boundaries_gdf.plot()

In [ ]:
landuse_clip = gpd.overlay(landuse_sg, park_boundaries_gdf, how='intersection')
landuse_clip['area_sqm'] = landuse_clip.geometry.area
landuse_clip.to_file("gis_files/landuse_clip_parks_allthree.geojson", driver='GeoJSON')

In [ ]:
for park in ['amk', 'jurong', 'punggol']:
    print("the ratio of blue to green space in", park, "is:")
    (landuse_clip[(landuse_clip['layer'] == park) & (landuse_clip['LU_DESC'] == 'PARK')]['area_sqm'].sum()) / (landuse_clip[(landuse_clip['layer'] == park) & (landuse_clip['LU_DESC'] == 'WATERBODY')]['area_sqm'].sum())

In [ ]:
for park in ['amk', 'jurong', 'punggol']:
    print("the ratio of blue to green space in", park, "is:")
    print((landuse_clip[(landuse_clip['layer'] == park) & (landuse_clip['LU_DESC'] == 'WATERBODY')]['area_sqm'].sum())/(landuse_clip[(landuse_clip['layer'] == park) & (landuse_clip['LU_DESC'] == 'PARK')]['area_sqm'].sum()))

In [ ]:
from shapely.ops import unary_union
for park in ['amk', 'jurong', 'punggol']:
    print("the total length of park boundary touching waterbody in", park, "is:")
    parks = landuse_clip[(landuse_clip['layer'] == park) & (landuse_clip['LU_DESC'] == 'PARK')].copy()
    water = landuse_clip[(landuse_clip['layer'] == park) & (landuse_clip['LU_DESC'] == 'WATERBODY')].copy()

    parks["geometry"] = parks.geometry.buffer(0)
    water["geometry"] = water.geometry.buffer(0)

    water_boundary = unary_union(water.geometry).boundary

    parks["water_touch_m"] = parks.geometry.boundary.intersection(water_boundary).length
    parks["water_touch_m"] = parks["water_touch_m"].astype(float)
    print(parks["water_touch_m"].sum())

In [ ]:
# --- Load GeoJSON ---
gdf = gpd.read_file("gis_files/landuse/landuse_1km_buffer_park.geojson")

# Expected columns: 'layer' (park name), 'LandUse' (category), 'area' (area of polygon)
assert {'layer', 'LandUse', 'area'}.issubset(gdf.columns), "Missing required columns"

In [ ]:
gdf.plot()

In [ ]:
# --- Compute total area per park ---
total_area = gdf.groupby('layer')['area'].sum().rename('total_area')

# Merge total area back
gdf = gdf.merge(total_area, on='layer')

# --- Compute proportion p_i for each LandUse type within each park ---
gdf['p'] = gdf['area'] / gdf['total_area']

gdf.head()

In [ ]:
# --- Compute Shannon entropy per park ---
def shannon_entropy(subdf: pd.DataFrame) -> float:
    p = subdf['p'].values
    p = p[p > 0]
    H = -np.sum(p * np.log(p))
    # normalised by ln(k)
    k = len(p)
    return H / np.log(k) if k > 1 else 0.0

entropy_df = gdf.groupby('layer').apply(shannon_entropy).reset_index()
entropy_df.columns = ['layer', 'Normalized_Shannon_Entropy']


# --- Optional: save result ---
entropy_df.to_csv("landuse_entropy.csv", index=False)
entropy_df

In [ ]:
# calculate the residentice % of each park
residence_per = gdf[gdf['LandUse'] == 'Residential'].groupby('layer')['p'].sum().reset_index()
residence_per.columns = ['layer', 'Residential_Percentage']
residence_per